In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 10),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import rateslib as rl
import QuantLib as ql

import datetime
import pytz

NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
LDN_tz = pytz.timezone("Europe/London") 
UTC_tz = pytz.timezone("UTC") 

import sys
sys.path.append("../")

from RVUtils.plt_timeseries import make_secondary_axis_plot

In [2]:
from MDP.USTFutures.USTFutureOptionMDP import USTFutureOptionMDP
from MDP.IRSwaptions.IRSwaptionMDP import IRSwaptionMDP

from TB.TimeseriesBuilder import TimeseriesBuilder
from TB.USTFutureOptionsTB import USTFutureOptionsTB 
from TB.IRSwaptionsTB import IRSwaptionsTB 

from Query.USTFutureOptions import USTFutureOptionQuery, USTFutureOptionStructure, USTFutureOptionValue
from Query.IRSwaptions import IRSwaptionQuery, IRSwaptionStructure, IRSwaptionValue

from BT.misc import ql_cal_date_range

In [6]:
# otc_listed_vol_tb = TimeseriesBuilder(
#     irswaptions_tb=IRSwaptionsTB(
#         IRSwaptionMDP(
#             source="GSQUANT-QL",
#             curve_source="ERIS_EOD_LIVE-QL_BASIC",
#         ),
#         show_tqdm=True,
#     ),
#     ustfutureoptions_tb=USTFutureOptionsTB(USTFutureOptionMDP(source="USTFO_DUAL-QL"), show_tqdm=True),
#     # ustfutureoptions_tb=USTFutureOptionsTB(USTFutureOptionMDP(source="BARCHART_USTFO-QL"), show_tqdm=True),
# )

In [7]:
# start = datetime.date(2025, 3, 15)
# end = datetime.date(2026, 3, 9)

# queries = [
    # USTFutureOptionQuery(symbol="TY_30|ATMS", value=USTFutureOptionValue.IV_NORMAL_BPS),
    # USTFutureOptionQuery(symbol="US_30|ATMS", value=USTFutureOptionValue.IV_NORMAL_BPS),

    # USTFutureOptionQuery(symbol="TYM26|ATMS", value=USTFutureOptionValue.IV_NORMAL_BPS),
    # IRSwaptionQuery(
    #     curve="USD-SOFR-1D",
    #     shorthand="1m7y",
    #     structure=IRSwaptionStructure.STRADDLE,
    #     strike="ATMF",
    #     value=IRSwaptionValue.NVOL,
    # ),
# ]

# df = otc_listed_vol_tb.get_timeseries(
#     start=start,
#     end=end,
#     queries=queries,
#     drop_multilevel_cols=True,
#     # ignore_cache=True
# )
# df

In [8]:
# plot, fig, ax, ax2, legend = make_secondary_axis_plot()
# plot(df["TY_30|ATMS OUTRIGHT IV_NORMAL_BPS"], which="left")
# # plot(df["USD-SOFR-1D 1Mx7Y STRADDLE BUY ATMF NVOL"], which="left")
# legend(show_date=True, loc="lower left")
# plt.show()

In [14]:
# python MDP/USTFutures/warm_ustfo_cache_parallel.py --start 2025-01-06 --end 2025-01-10 --chunks 5 --workers 5 --heartbeat-seconds 30 --report-json cache/ustfo_cache_warm_report.json

In [11]:
start = datetime.date(2025, 4, 6)
end = datetime.date(2025, 4, 10)

dates = ql_cal_date_range(ql_cal=ql.UnitedStates(ql.UnitedStates.GovernmentBond), start=start, end=end, to_date=True)
dates

[datetime.date(2025, 4, 7),
 datetime.date(2025, 4, 8),
 datetime.date(2025, 4, 9),
 datetime.date(2025, 4, 10)]

In [13]:
ustfo_qs_mdp = USTFutureOptionMDP(source="USTFO_DUAL-QL")

errors = []
for d in dates:
    # for cmt in ["07", "14", "30", "60", "90"]:
    for cmt in ["30", "60", "90"]:
        try:
            out = ustfo_qs_mdp.get_pricer(
                {
                    "endpoint": "option_snapshot",
                    "symbols": [f"TU_{cmt}|ATMS", f"FV_{cmt}|ATMS", f"TY_{cmt}|ATMS", f"TN_{cmt}|ATMS", f"US_{cmt}|ATMS", f"UL_{cmt}|ATMS"],
                    "timestamp": d, 
                    "show_tqdm": True,
                    "use_ql_calculator": True,
                }
            )
            print(d, out)

            smiles = ustfo_qs_mdp.fetch_bulk_sabr_smile(
                {
                    "globex_symbols": [f"TU_{cmt}", f"FV_{cmt}", f"TY_{cmt}", f"TN_{cmt}", f"US_{cmt}", f"UL_{cmt}"],
                    "timestamps": [d],
                    "show_tqdm": True,
                }
            )
            print(d, smiles)

        except Exception as e:
            errors.append({"date": d, "cmt": cmt, "error": str(e)})

2025-04-07 {'TY_30|ATMS': [QLUSTFutureOptionPricer(_symbol='TY_30|ATMS', _right='S', _underlying_symbol='ZNM25', _strike=112.0, _quote_timestamp=datetime.datetime(2025, 4, 7, 0, 0, tzinfo=<DstTzInfo 'America/New_York' EDT-1 day, 20:00:00 DST>), _expiry_date=datetime.date(2025, 5, 7), _market_price=2.192074854939581, _model_price=2.192074854939581, _iv_normal=9.615328196301466, _delta=0.018023119859294834, _gamma=0.28832050917107976, _vega=0.22785997162230234, _theta=-13.23157290222218, _forward=112.0625, _discount=0.996380928883563, _fv01=0.06488097469662428, _meta_data={'schema': 1, 'source': 'SYNTH_STRADDLE', 'symbol': 'TY_30|ATMS', 'underlying_symbol': 'ZNM25', 'legs': ['ZNM25|1120C@2025-04-07T00:00:00-04:00', 'ZNM25|1120P@2025-04-07T00:00:00-04:00'], 'curve_name': 'USD-SOFR-1D-Q12xM12STIRT', 'curve_error': None, 'vendor_legs': [None, None]})], 'TU_30|ATMS': [QLUSTFutureOptionPricer(_symbol='TU_30|ATMS', _right='S', _underlying_symbol='ZTM25', _strike=104.0, _quote_timestamp=datetim